<a href="https://colab.research.google.com/github/bbberylll/BDA/blob/main/09_12_%EC%84%B8%EC%85%98_%EB%AA%A8%EB%8D%B8%ED%9B%88%EB%A0%A8_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **모델 훈련 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch04 연습문제 1, 5, 9, 10
- 개념 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

### **1. 수백만 개의 특성을 가진 훈련 세트에서는 어떤 선형 회귀 알고리즘을 사용할 수 있을까요?**
___


확률적 경사 하강법 & 미니 배치 경사 하강법을 이용해야 한다. 전체 훈련 세트를 이용하는 것보다 속도가 더 빠르다.

만약 일반적인 경사 하강법을 사용한다면 메모리 공간이 부족할 가능성이 높다

### **2. 배치 경사 하강법을 사용하고 에포크마다 검증 오차를 그래프로 나타내봤습니다. 검증 오차가 일정하게 상승되고 있다면 어떤 일이 일어나고 있는 걸까요? 이 문제를 어떻게 해결할 수 있나요?**
___

에포크마다 검증 오차가 일정하게 상승하고 있다면 학습률이 너무 큰 상황일 수 있다.
훈련 오차를 별도로 확인해 동일한 모습을 보인다면 학습률을 지속적으로 줄이는 방향으로 문제를 해결 할 수 있을 것이다.

만약 훈련 오차에서 이러한 모습이 나타나지 않는다면 모델의 overfitting을 의심할 수 있다. 따라서 학습을 조기 종료하고 모델을 새롭게 학습시켜야 한다.

### **3. 릿지 회귀를 사용했을 때 훈련 오차가 검증 오차가 거의 비슷하고 둘 다 높았습니다. 이 모델에는 높은 편향이 문제인가요, 아니면 높은 분산이 문제인가요? 규제 하이퍼파라미터 $\alpha$를 증가시켜야 할까요 아니면 줄여야 할까요?**
___

릿지 회귀의 경우 알파 값이 너무 작으면 과적합이 발생할 수 있음 (penalty가 줄어드는 거니까 규제가 줄어들면 과적합이 발생).

훈련 오차와 검증 오차가 모두 높은 상황은 과소적합이 발생한 상황인 것. 따라서 높은 편향이 문제인 것이다.
따라서 과소적합을 해결하기 위해 모델의 복잡성을 키워야 함
-- 규제 하이퍼파라미터를 증가시켜야 한다.

### **4. 다음과 같이 사용해야 하는 이유는?**
___
- 평범한 선형 회귀(즉, 아무런 규제가 없는 모델) 대신 릿지 회귀
- 릿지 회귀 대신 라쏘 회귀
- 라쏘 회귀 대신 엘라스틱넷

평범한 선형 회귀는 규제가 없기 때문에 다항 회귀에서는 과적합될 가능성이 높다.
따라서 평범한 선형회귀보다는 릿지 회귀가 더 적절하다.

사용되는 특성의 수가 적어보일 경우에는 상관 변수가 많을 때 릿지보다는 라쏘와 엘라스틱넷이 더 안정적인 성향을 보이며,

라쏘보다는 엘라스틱넷이 다중공선성에 더 강한 모습을 보인다.


### **추가) 조기 종료를 사용한 배치 경사 하강법으로 iris 데이터를 활용해 소프트맥스 회귀를 구현해보세요(사이킷런은 사용하지 마세요)**


---



In [ ]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()

X = iris['data'][:,(2,3)]
y = (iris['target'])


X_with_bias = np.c_[np.ones([len(X),1]),X]

np.random.seed(2042)

test_ratio = 0.2
validation_ratio = 0.2
total_size=len(X_with_bias)

test_size = int(total_size*test_ratio)
validation_size= int(total_size*validation_ratio)
train_size = total_size - test_size - validation_size

rnd_indices = np.random.premutation(total_size)
X_train = X_with_bias[rnd_indices[:train_size]]
y_train = y[rnd_indices[:train_size]]

X_valid = X_with_bias[rnd_indices[train_size:-test_size]]
y_valid = y[rnd_indices[train_size:-test_size]]

X_test = X_with_bias[rnd_indices[-test_size:]]
y_test = y[rnd_indices[-test_size:]]

## 원핫 인코딩...은 np를 이용하는 방법만 생각이 났습니다,,,

def softmax(logits):
    exps = np.exp(logits)
    exp_sums = np.sum(exps, axis=1, keepdims=True)
    return exps / exp_sums

## 샘플 x에 대한 각 클래스의 점수가 주어졌을 때
## 샘플이 클래스 k에 속할 추정 확률을 구하는 함수까지 구현했습니다

eta = 0.1
n_iterations = 5001
m = len(X_train)
epsilon = 1e-7
alpha = 0.1  # 규제 하이퍼파라미터
best_loss = np.infty

n_inputs = X_train.shape[1]
n_outputs = len(np.unique(y_train))

Theta = np.random.randn(n_inputs, n_outputs)

for iteration in range(n_iterations):
    logits = X_train.dot(Theta)
    Y_proba = softmax(logits)
    error = Y_proba - ##y_train을 원핫인코딩한 결과
    gradients = 1/m * X_train.T.dot(error) + np.r_[np.zeros([1, n_outputs]), alpha * Theta[1:]]
    Theta = Theta - eta * gradients

    logits = X_valid.dot(Theta)
    Y_proba = softmax(logits)
    xentropy_loss = -np.mean(np.sum(##y의 valid를 원핫인코딩한결과
                                    * np.log(Y_proba + epsilon), axis=1))
    l2_loss = 1/2 * np.sum(np.square(Theta[1:]))
    loss = xentropy_loss + alpha * l2_loss
    if iteration % 500 == 0:
        print(iteration, loss)
    if loss < best_loss:
        best_loss = loss
    else:
        print(iteration - 1, best_loss)
        print(iteration, loss, "조기 종료!")
        break